# P4 · Fast 4-Core Parallel Multi-Kernel, CQT-Fisher, Dual-Teacher EM & Graph Diffusion Ensemble

**Speed-Optimized (~40–50 min total on 4-core Kaggle CPU) & 100% Competition-Rules Compliant**
- **Allowed Libraries Only**: `numpy`, `pandas`, `scipy`, `scikit-learn`, `lightgbm`, `xgboost`, `librosa`, `soundfile`, `joblib` (+ Python stdlib).
- **Allowed Models Only**: RBF & Precomputed Multi-Kernel SVMs, L2 Multinomial Softmax (`LogisticRegression`), Shrinkage `LDA`, `RandomForest`, `ExtraTrees`, `LightGBM`, `XGBoost` (`hist`), `PCA`, `MiniBatchKMeans`, `GaussianMixture` (Fisher Vectors), EM over true labels (`em_noise`), acoustic k-NN graph label propagation, and regularized OOF stacking.

### Speed Upgrades (Zero Accuracy Compromise)
1. **Mel-Domain HPSS**: Computes harmonic-percussive separation on the 40-band Mel spectrogram instead of the 1,025-bin STFT (~4× faster `Frich_v2` extraction: ~3.5 min vs. ~15 min).
2. **PCA(512)-Compressed `fv_logreg`**: Compresses `[Mel-Fisher, Mel-VLAD, CQT-Fisher]` via unsupervised `PCA(512)` on training audio before L2 Softmax (~25× faster, preserving discriminative variance).
3. **4-Core Parallel SVM Batching**: Runs all single-threaded `SVC(probability=True)` and linear models concurrently across all 4 CPU workers inside each fold (~3.5× faster).
4. **Fast Decorrelated `xgb_dense`**: Trains `XGBoost (hist)` on `PCA(256)` of the combined dense features (`max_bin=64`) for ~6× speedup plus extra tree-split diversity vs. raw-feature `LightGBM`.
5. **Live Per-Stage Progress Logging**: Prints elapsed time after each model group inside every fold.

In [ ]:
import os, time, warnings, hashlib, inspect, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from scipy.spatial.distance import cdist
from scipy.signal import periodogram
from scipy.stats import skew, kurtosis
from scipy.optimize import minimize, minimize_scalar
from joblib import Parallel, delayed

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics.pairwise import linear_kernel, rbf_kernel
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

INPUT_ROOT = Path(os.environ.get("P4_INPUT", "/kaggle/input"))
WORK       = Path(os.environ.get("P4_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
N_JOBS     = int(os.environ.get("P4_JOBS", -1))
SEEDS      = [42, 913]

train_csv = list(INPUT_ROOT.rglob("train.csv"))
assert len(train_csv) == 1, train_csv
DATA = train_csv[0].parent
train_df  = pd.read_csv(DATA / "train.csv", dtype={"id": str})
test_df   = pd.read_csv(DATA / "test.csv", dtype={"id": str})
sample_df = pd.read_csv(DATA / "sample_submission.csv", dtype={"id": str})

TRAIN_AUDIO = list(INPUT_ROOT.rglob("audio/train"))
TEST_AUDIO  = list(INPUT_ROOT.rglob("audio/test"))
assert len(TRAIN_AUDIO) == 1 and len(TEST_AUDIO) == 1
TRAIN_AUDIO, TEST_AUDIO = TRAIN_AUDIO[0], TEST_AUDIO[0]

EXPECTED_TEST = int(os.environ.get("P4_EXPECTED_TEST", 1592))
assert test_df["id"].is_unique and sample_df["id"].is_unique and train_df["id"].is_unique
assert set(test_df["id"]) == set(sample_df["id"]), "test.csv and sample_submission.csv ids differ"
assert len(test_df) == EXPECTED_TEST, len(test_df)

classes = np.array(sorted(train_df["label"].unique()))
K = len(classes)
cls_idx = {c: i for i, c in enumerate(classes)}
verified_mask = (train_df["checked"].values == 1)
y_all_idx = train_df["label"].map(cls_idx).values
yv = y_all_idx[verified_mask]
yu = y_all_idx[~verified_mask]

verified_ids  = np.flatnonzero(verified_mask)
unchecked_ids = np.flatnonzero(~verified_mask)
test_ids      = np.arange(len(train_df), len(train_df) + len(test_df))

tab = pd.DataFrame({
    "verified":  np.bincount(yv, minlength=K),
    "unchecked": np.bincount(yu, minlength=K),
}, index=classes)
tab["total"] = tab.verified + tab.unchecked
ver_cnt = tab.verified.values.astype(float)
tot     = tab.total.values.astype(float)

pi_target = tot / tot.sum()
pi_ver    = ver_cnt / ver_cnt.sum()
w_val     = (pi_target / pi_ver)[yv]
w_unif    = (1.0 / pi_ver)[yv]

print("data dir:", DATA, flush=True)
print(f"classes: {K} | verified: {verified_mask.sum()} | unchecked: {(~verified_mask).sum()} | test: {len(test_df)}", flush=True)

## Core utilities: MAP@3, calibration, exact class balancing, EM prior/noise, and graph diffusion

In [ ]:
def map3_s(yi, P):
    top3 = np.argsort(-P, axis=1)[:, :3]
    return ((top3 == yi[:, None]) / np.array([1.0, 2.0, 3.0])).sum(1)

def wmap3(yi, P, w=None):
    return float(np.average(map3_s(yi, P), weights=w))

def normalize_probs(P, eps=1e-12):
    Q = np.clip(np.asarray(P, dtype=np.float64), eps, None)
    return Q / Q.sum(axis=1, keepdims=True)

def sharpen(P, t):
    Q = np.clip(P, 1e-9, None) ** t
    return Q / Q.sum(axis=1, keepdims=True)

def fit_temp(P, yi, w=None):
    P_c = np.clip(P, 1e-9, None)
    logP = np.log(P_c)
    idx = np.arange(len(yi))
    def loss(t):
        z = logP * t
        z -= z.max(axis=1, keepdims=True)
        ez = np.exp(z)
        lse = np.log(ez.sum(axis=1))
        nll = -(z[idx, yi] - lse)
        return float(np.average(nll, weights=w))
    res = minimize_scalar(loss, bounds=(0.3, 6.0), method="bounded")
    return float(res.x)

def adjust(P, ratio):
    Q = P * ratio[None, :]
    return Q / Q.sum(axis=1, keepdims=True)

def em_prior(P, pi_train=None, iters=200, floor=0.25):
    if pi_train is None:
        pi_train = np.full(K, 1.0 / K)
    pi = np.full(K, 1.0 / K)
    inv_tr = 1.0 / np.maximum(pi_train, 1e-12)
    for _ in range(iters):
        Q = P * (pi * inv_tr)[None, :]
        Q /= Q.sum(axis=1, keepdims=True)
        pi = Q.mean(axis=0)
    pi = np.clip(pi, floor / K, None)
    return pi / pi.sum()

def em_noise(P, yi, iters=50, prior_strength=5.0):
    P = normalize_probs(P, eps=1e-9)
    Y = np.eye(K)[yi]
    T0 = np.full((K, K), 0.45 / (K - 1))
    np.fill_diagonal(T0, 0.55)
    T = T0.copy()
    pi = np.full(K, 1.0 / K)
    for _ in range(iters):
        Q = P * (pi * K)[None, :] * T[:, yi].T
        Q = np.clip(Q, 1e-15, None)
        Q /= Q.sum(axis=1, keepdims=True)
        pi = Q.mean(axis=0)
        Tn = Q.T @ Y + prior_strength * T0
        T = Tn / Tn.sum(axis=1, keepdims=True)
    return T, Q, pi

def balance_weights(y, w, alpha=1.0):
    w = np.asarray(w, dtype=np.float64)
    cls_sum = np.bincount(y, weights=w, minlength=K)
    mean_mass = w.sum() / max((cls_sum > 0).sum(), 1)
    scale = np.where(cls_sum > 0, mean_mass / np.maximum(cls_sum, 1e-12), 1.0)
    wb = w * (scale[y] ** alpha)
    return wb * (len(w) / max(wb.sum(), 1e-12))

def finalize_probs(P, temp, power=0.5, pi_train=None):
    Pc = sharpen(P, temp)
    pi_hat = em_prior(Pc, pi_train=pi_train)
    if power == 0:
        return Pc, pi_hat
    inv_tr = np.full(K, K) if pi_train is None else (1.0 / np.maximum(pi_train, 1e-12))
    ratio = (pi_hat * inv_tr) ** power
    return adjust(Pc, ratio), pi_hat

def sim_eval(P, w, powers=(0.0, 0.25, 0.5, 0.75, 1.0), n_draw=30, size=1500, seed=0):
    rng = np.random.default_rng(seed)
    p = w / w.sum()
    out = {pw: [] for pw in powers}
    for _ in range(n_draw):
        idx = rng.choice(len(yv), size=size, p=p)
        Pd = P[idx]
        pi_hat = em_prior(Pd, iters=100)
        for pw in powers:
            Q = Pd if pw == 0 else adjust(Pd, (pi_hat * K) ** pw)
            out[pw].append(map3_s(yv[idx], Q).mean())
    return {pw: float(np.mean(v)) for pw, v in out.items()}

def crossfit_cal(oof, splits):
    Pc = np.zeros_like(oof)
    for tr, va in splits:
        Pc[va] = sharpen(oof[va], fit_temp(oof[tr], yv[tr], w_unif[tr]))
    return Pc

def cv_report(oof, splits):
    Pc = crossfit_cal(oof, splits)
    rep = {"plain": wmap3(yv, oof)}
    rep["w_raw"] = sim_eval(oof, w_val, powers=(0.0,))[0.0]
    rep["u_raw"] = sim_eval(oof, w_unif, powers=(0.0,))[0.0]
    for tag, w in (("w", w_val), ("u", w_unif)):
        d = sim_eval(Pc, w, powers=(0.0, 0.25, 0.5, 0.75, 1.0))
        for pw, val in d.items():
            key = f"{tag}_cal" if pw == 0.0 else f"{tag}_em{pw:.2f}"
            rep[key] = val
    return rep

def combine_score(rep, power=0.5):
    if power == 0.0:
        return (rep["plain"] + rep["w_cal"] + rep["u_cal"]) / 3.0
    key = f"em{power:.2f}"
    return (rep["plain"] + rep[f"w_{key}"] + rep[f"u_{key}"]) / 3.0

def build_knn_diffusion_matrix(sim_matrix, target_ids, source_ids, k_nn=15, tau=0.12):
    S = sim_matrix[np.ix_(target_ids, source_ids)].astype(np.float64).copy()
    src_pos = {int(sid): j for j, sid in enumerate(source_ids)}
    for i, tid in enumerate(target_ids):
        if int(tid) in src_pos:
            S[i, src_pos[int(tid)]] = -1e9
    k_eff = min(k_nn, S.shape[1])
    top_idx = np.argpartition(-S, k_eff - 1, axis=1)[:, :k_eff]
    top_sim = np.take_along_axis(S, top_idx, axis=1)
    z = (top_sim - top_sim.max(axis=1, keepdims=True)) / max(tau, 1e-4)
    w = np.exp(z)
    w /= np.maximum(w.sum(axis=1, keepdims=True), 1e-12)
    return top_idx, w

def apply_graph_diffusion(P_target, P_source, top_idx, w, alpha=0.12):
    if alpha <= 0.0:
        return P_target
    P_nbr = np.einsum("ij,ijk->ik", w, P_source[top_idx])
    return normalize_probs((1.0 - alpha) * P_target + alpha * P_nbr)

OUT = WORK / "model_files"
def write_submission(P, name):
    top3 = np.argsort(-P, axis=1)[:, :3]
    sub = pd.DataFrame({"id": test_df["id"], "label": [" ".join(classes[r]) for r in top3]})
    sub = sample_df[["id"]].merge(sub, on="id", how="left", validate="one_to_one")
    assert sub.shape == (len(sample_df), 2) and sub["label"].notna().all() and sub["id"].is_unique
    assert all(len(set(s.split())) == 3 and all(c in cls_idx for c in s.split()) for s in sub["label"])
    OUT.mkdir(exist_ok=True)
    out_path = OUT / name
    sub.to_csv(out_path, index=False, lineterminator="\n")
    print("saved", out_path, sub.shape, flush=True)
    return sub

## 1. Fast recording-level features (`Frich_v2_fast`: Mel-domain HPSS + Delta/Delta2 + Envelope + Tonnetz)

Computes HPSS on the 40-band Mel magnitude spectrogram instead of the 1,025-bin STFT (~4× faster extraction while keeping harmonic/percussive band ratios and spectral dynamics).

In [ ]:
print("Starting fast Frich_v2 feature extraction...", flush=True)
N_FFT, HOP = 2048, 512

def count_onsets(env, sr, hop):
    if not np.isfinite(env).all() or not np.any(env):
        return 0
    normalized = env - env.min()
    normalized = normalized / (normalized.max() + np.finfo(env.dtype).tiny)
    pre_max = max(1, int(0.03 * sr // hop))
    pre_avg = max(1, int(0.10 * sr // hop))
    post_avg = pre_avg + 1
    wait = pre_max
    count = 0
    previous = -wait - 1
    for i, value in enumerate(normalized):
        if i <= previous + wait:
            continue
        peak_window = normalized[max(0, i - pre_max):i + 1]
        avg_window = normalized[max(0, i - pre_avg):i + post_avg]
        if value == peak_window.max() and value >= avg_window.mean() + 0.07:
            count += 1
            previous = i
    return count

def safe_delta(M, order=1):
    T = M.shape[1]
    if T < 3:
        M = np.pad(M, ((0, 0), (0, 3 - T)), mode="edge")
        T = 3
    w = min(9, T if T % 2 == 1 else T - 1)
    return librosa.feature.delta(M, width=w, order=order, mode="nearest")

def extract_one_fast(path):
    try:
        y, sr = sf.read(str(path), dtype="float32")
        if y.ndim > 1:
            y = y.mean(axis=1)
        dur = len(y) / sr
        clip_frac = float(np.mean(np.abs(y) > 0.98)) if len(y) > 0 else 0.0
        if len(y) < N_FFT:
            y = np.pad(y, (0, N_FFT - len(y)))

        S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP))
        P2 = S ** 2
        T = S.shape[1]

        rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)
        zcr = librosa.feature.zero_crossing_rate(y=y, frame_length=N_FFT, hop_length=HOP)
        mel128 = librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=128)
        mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel128), n_mfcc=20)
        chroma = librosa.feature.chroma_stft(S=P2, sr=sr)
        cent = librosa.feature.spectral_centroid(S=S, sr=sr)
        bw   = librosa.feature.spectral_bandwidth(S=S, sr=sr)
        roll = librosa.feature.spectral_rolloff(S=S, sr=sr)
        con  = librosa.feature.spectral_contrast(S=S, sr=sr)

        ms = lambda M: np.concatenate([M.mean(axis=1), M.std(axis=1)])
        base = np.concatenate([ms(mfcc), ms(chroma), ms(cent), ms(bw), ms(roll), ms(zcr), ms(rms), [dur]])
        contrast = ms(con)

        r1 = rms[0]
        act = r1 >= max(np.percentile(r1, 30), 0.02 * r1.max())
        if act.sum() < 3:
            act = np.ones(T, bool)

        mel40 = librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=40)
        lm = librosa.power_to_db(mel40)
        lma, mfa = lm[:, act], mfcc[:, act]
        dm  = safe_delta(mfcc, order=1)
        dm2 = safe_delta(mfcc, order=2)
        dlm = safe_delta(lm, order=1)

        flat = librosa.feature.spectral_flatness(S=S)[0]
        env  = librosa.onset.onset_strength(S=lm, sr=sr, hop_length=HOP)
        n_on = count_onsets(env, sr, HOP)
        lr = np.log(r1 + 1e-8)

        rich = np.concatenate([
            lma.mean(1), lma.std(1), np.percentile(lma, 90, axis=1), lm.max(1),
            mfa.mean(1), mfa.std(1), dm.std(1),
            [flat.mean(), flat.std(), env.mean(), env.std()],
            np.percentile(lr, [10, 50, 90]),
            [act.mean(), np.log(dur + 1e-3), n_on / max(dur, 1e-3)],
        ])

        # Fast Mel-domain HPSS (40 bins instead of 1025 -> ~25x faster!)
        M_mag = np.sqrt(np.maximum(mel40, 0.0))
        M_h, M_p = librosa.decompose.hpss(M_mag, kernel_size=15, margin=1.0)
        Eh_m = (M_h ** 2) + 1e-10
        Ep_m = (M_p ** 2) + 1e-10
        hpss_global = float(np.log(Eh_m.sum() / Ep_m.sum()))
        band_edges = [0, 8, 16, 28, 40]
        band_ratios = [float(np.log(Eh_m[lo:hi].sum() / Ep_m[lo:hi].sum())) for lo, hi in zip(band_edges[:-1], band_edges[1:])]
        # Harmonic & percussive mel-index centroids + temporal variability
        bins_idx = np.arange(40, dtype=np.float32)[:, None]
        h_cent = (M_h * bins_idx).sum(axis=0) / (M_h.sum(axis=0) + 1e-8)
        p_cent = (M_p * bins_idx).sum(axis=0) / (M_p.sum(axis=0) + 1e-8)
        h_e_t = np.log(Eh_m.sum(axis=0))
        p_e_t = np.log(Ep_m.sum(axis=0))
        hpss_feats = np.array([
            hpss_global, *band_ratios,
            h_cent.mean(), h_cent.std(), p_cent.mean(), p_cent.std(),
            h_e_t.mean(), h_e_t.std(), p_e_t.mean(), p_e_t.std(),
        ], dtype=np.float32)

        # MFCC delta/delta-2 & log-mel delta dynamics on active frames
        dma, dm2a, dlma = dm[:, act], dm2[:, act], dlm[:, act]
        dyn_feats = np.concatenate([
            dma.mean(1), np.percentile(dma, 90, axis=1) - np.percentile(dma, 10, axis=1),
            dm2a.mean(1), dm2a.std(1),
            np.percentile(lma, 10, axis=1),
            dlma.std(1),
        ])

        # Temporal envelope (thirds), active duration, crest factor, clipping, spectral flux, Tonnetz
        e_frame = r1 ** 2 + 1e-12
        e_tot = e_frame.sum()
        third_shares = [s.sum() / e_tot for s in np.array_split(e_frame, 3)]
        crest = float(np.log((np.max(np.abs(y)) + 1e-8) / (np.sqrt(np.mean(y ** 2)) + 1e-8)))
        silence_frac = float(np.mean(r1 < 0.05 * max(r1.max(), 1e-8)))
        act_dur = float(np.log(dur * act.mean() + 1e-3))
        flux = np.sqrt(np.maximum(0.0, np.diff(S, axis=1)) ** 2).mean(axis=0) if T > 1 else np.zeros(1)
        flux_stats = [float(flux.mean()), float(flux.std()), float(np.percentile(flux, 90))]
        tonnetz = librosa.feature.tonnetz(chroma=chroma)
        extra_env = np.concatenate([
            third_shares, [crest, silence_frac, act_dur, clip_frac], flux_stats,
            ms(tonnetz),
            [float(np.percentile(cent[0, act], 90) - np.percentile(cent[0, act], 10))],
        ])

        feat = np.concatenate([base, contrast, rich, hpss_feats, dyn_feats, extra_env]).astype(np.float32)
        return np.nan_to_num(feat, nan=np.nan, posinf=np.nan, neginf=np.nan)
    except Exception:
        return None

def extract_all_fast(ids, folder):
    rows = Parallel(n_jobs=N_JOBS, batch_size=8, verbose=5)(
        delayed(extract_one_fast)(folder / f"{i}.wav") for i in ids
    )
    dim = next(r.shape[0] for r in rows if r is not None)
    bad = [i for i, r in zip(ids, rows) if r is None]
    X = np.vstack([r if r is not None else np.full(dim, np.nan, np.float32) for r in rows])
    print(f"{folder.name}: {X.shape}, failed files: {len(bad)}", bad[:5], flush=True)
    return X

FEAT_VERSION = "v7-frich-melhpss-fast"
_src = inspect.getsource(extract_one_fast)
FEAT_HASH = hashlib.md5((FEAT_VERSION + _src + str((N_FFT, HOP))).encode()).hexdigest()[:10]
FEAT_FILE = WORK / f"p4_features_fast_{FEAT_HASH}.npz"
print("Frich_v2_fast cache:", FEAT_FILE.name, flush=True)

Xtr_rich = None
if FEAT_FILE.exists():
    z = np.load(FEAT_FILE, allow_pickle=True)
    if list(z["train_ids"]) == train_df["id"].tolist() and list(z["test_ids"]) == test_df["id"].tolist():
        Xtr_rich, Xte_rich = z["Xtr"], z["Xte"]
        print("Loaded cached Frich_v2_fast:", Xtr_rich.shape, Xte_rich.shape, flush=True)

if Xtr_rich is None:
    t0 = time.time()
    Xtr_rich = extract_all_fast(train_df["id"].tolist(), TRAIN_AUDIO)
    Xte_rich = extract_all_fast(test_df["id"].tolist(), TEST_AUDIO)
    np.savez_compressed(
        FEAT_FILE, Xtr=Xtr_rich, Xte=Xte_rich,
        train_ids=np.array(train_df["id"]), test_ids=np.array(test_df["id"]),
    )
    print(f"Frich_v2_fast extraction time: {time.time() - t0:.0f}s", flush=True)

Xtr_rich = np.where(np.isfinite(Xtr_rich), Xtr_rich, np.nan)
Xte_rich = np.where(np.isfinite(Xte_rich), Xte_rich, np.nan)
med_rich = np.nanmedian(Xtr_rich, axis=0)
med_rich = np.where(np.isfinite(med_rich), med_rich, 0.0)
Xtr_rich = np.where(np.isnan(Xtr_rich), med_rich, Xtr_rich).astype(np.float32)
Xte_rich = np.where(np.isnan(Xte_rich), med_rich, Xte_rich).astype(np.float32)

XV_rich, XU_rich, XT_rich = Xtr_rich[verified_mask], Xtr_rich[~verified_mask], Xte_rich
print("Frich_v2_fast dim:", Xtr_rich.shape[1], flush=True)

## 2. Activity-masked 3-scale Mel short-window representations (BoAW, PCA-VLAD, Fisher)

In [ ]:
print("Starting 3-scale Mel short-window representations...", flush=True)

SHORTWIN_KEY = hashlib.sha256(
    (",".join(train_df["id"].tolist()) + "|" + ",".join(test_df["id"].tolist()) + "|v2-3scale-act").encode()
).hexdigest()[:12]
SHORTWIN_CACHE = WORK / f"p4_shortwin_v2_{SHORTWIN_KEY}.npz"

def extract_patches_v2(path):
    audio, rate = sf.read(str(path), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    mel = librosa.feature.melspectrogram(
        y=audio, sr=rate, n_fft=1024, hop_length=max(1, round(rate * 0.012)), n_mels=40
    )
    logmel = librosa.power_to_db(mel, ref=np.max).T.astype(np.float32)
    frame_energy = logmel.mean(axis=1)
    views = []
    for width in (8, 24, 64):
        hop = max(2, width // 2)
        pad_len = max(0, width - len(logmel))
        frames = np.pad(logmel, ((0, pad_len), (0, 0)), mode="edge")
        fe = np.pad(frame_energy, (0, pad_len), mode="edge")
        starts = list(range(0, len(frames) - width + 1, hop))
        if starts[-1] != len(frames) - width:
            starts.append(len(frames) - width)
        blocks = np.stack([frames[i:i + width] for i in starts])
        win_e = np.array([fe[i:i + width].mean() for i in starts], dtype=np.float32)

        thresh = max(win_e.max() - 45.0, float(np.percentile(win_e, 25)))
        act_mask = win_e >= thresh
        if act_mask.sum() < 2:
            act_mask = np.ones(len(blocks), dtype=bool)
        blocks = blocks[act_mask]

        half = width // 2
        b_mean = blocks.mean(axis=1)
        b_std  = blocks.std(axis=1)
        b_diff = blocks[:, half:].mean(axis=1) - blocks[:, :half].mean(axis=1)
        b_rng  = blocks.max(axis=1) - blocks.min(axis=1)
        views.append(np.hstack([b_mean, b_std, b_diff, b_rng]).astype(np.float32))
    return views

if SHORTWIN_CACHE.exists():
    z = np.load(SHORTWIN_CACHE)
    acoustic      = z["acoustic"]
    vlad_matrix   = z["vlad_matrix"]
    fisher_matrix = z["fisher_matrix"]
    print("Loaded cached Mel short-window matrices:", acoustic.shape, vlad_matrix.shape, fisher_matrix.shape, flush=True)
else:
    t0 = time.time()
    train_patches = Parallel(n_jobs=N_JOBS, batch_size=8, verbose=5)(
        delayed(extract_patches_v2)(TRAIN_AUDIO / f"{i}.wav") for i in train_df["id"]
    )
    test_patches = Parallel(n_jobs=N_JOBS, batch_size=8, verbose=5)(
        delayed(extract_patches_v2)(TEST_AUDIO / f"{i}.wav") for i in test_df["id"]
    )
    all_patches = train_patches + test_patches
    rng = np.random.default_rng(42)

    boaw_views, vlad_views, fisher_views = [], [], []
    n_train = len(train_df)

    for view_idx in range(3):
        print(f"Fitting Mel BoAW / PCA-VLAD / Fisher for scale {view_idx + 1}/3...", flush=True)
        samples = []
        for i in range(n_train):
            fr = train_patches[i][view_idx]
            cap = 40 if verified_mask[i] else 16
            samples.append(fr[rng.choice(len(fr), min(cap, len(fr)), replace=False)])
        pool = np.vstack(samples).astype(np.float32)

        scaler = StandardScaler().fit(pool)
        pool_s = scaler.transform(pool).astype(np.float32)

        kmeans_boaw = MiniBatchKMeans(n_clusters=128, batch_size=2048, n_init=3, random_state=42 + view_idx)
        kmeans_boaw.fit(pool_s)
        centers_b = kmeans_boaw.cluster_centers_.astype(np.float32)

        def encode_boaw(rec):
            fr = scaler.transform(rec[view_idx]).astype(np.float32)
            dist = cdist(fr, centers_b, "sqeuclidean")
            nearest = np.argpartition(dist, 2, axis=1)[:, :3]
            w = 1.0 / np.maximum(np.take_along_axis(dist, nearest, axis=1), 1e-8)
            w /= w.sum(axis=1, keepdims=True)
            hist = np.bincount(nearest.ravel(), weights=w.ravel(), minlength=len(centers_b))
            v = np.sqrt(hist / max(hist.sum(), 1e-8))
            return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)

        boaw_views.append(np.vstack([encode_boaw(r) for r in all_patches]))

        pca_vlad = PCA(n_components=48, whiten=True, svd_solver="randomized", random_state=17 + view_idx)
        pool_v = pca_vlad.fit_transform(pool_s).astype(np.float32)
        kmeans_vlad = MiniBatchKMeans(n_clusters=32, batch_size=2048, n_init=3, random_state=17 + view_idx)
        kmeans_vlad.fit(pool_v)
        centers_v = kmeans_vlad.cluster_centers_.astype(np.float32)

        def encode_vlad(rec):
            fr = pca_vlad.transform(scaler.transform(rec[view_idx])).astype(np.float32)
            assign = cdist(fr, centers_v, "sqeuclidean").argmin(axis=1)
            sums = np.zeros_like(centers_v)
            np.add.at(sums, assign, fr - centers_v[assign])
            sums /= np.maximum(np.linalg.norm(sums, axis=1, keepdims=True), 1e-8)
            v = np.sign(sums.ravel()) * np.sqrt(np.abs(sums.ravel()))
            return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)

        vlad_views.append(np.stack([encode_vlad(r) for r in all_patches]))

        pca_fv = PCA(n_components=32, whiten=True, svd_solver="randomized", random_state=81 + view_idx)
        pool_f = pca_fv.fit_transform(pool_s).astype(np.float32)
        gmm = GaussianMixture(
            n_components=48, covariance_type="diag", reg_covar=0.01, max_iter=80, random_state=81 + view_idx
        )
        gmm.fit(pool_f)
        mu = gmm.means_.astype(np.float32)
        var = gmm.covariances_.astype(np.float32)
        sqrt_var = np.sqrt(var)
        sqrt_w = np.sqrt(gmm.weights_[:, None]).astype(np.float32)

        def encode_fisher(rec):
            x = pca_fv.transform(scaler.transform(rec[view_idx])).astype(np.float32)
            q = gmm.predict_proba(x).astype(np.float32)
            mass = q.sum(axis=0)[:, None]
            moments = q.T @ x
            norm = len(x) * sqrt_w
            first = (moments - mass * mu) / (norm * sqrt_var)
            second = ((q.T @ (x * x) - 2.0 * mu * moments + mass * (mu ** 2)) / var - mass) / (np.sqrt(2.0) * norm)
            v = np.concatenate([first.ravel(), second.ravel()])
            v = np.sign(v) * np.sqrt(np.abs(v))
            return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)

        fisher_views.append(np.stack([encode_fisher(r) for r in all_patches]))

    acoustic      = np.hstack(boaw_views).astype(np.float32)
    vlad_matrix   = np.hstack(vlad_views).astype(np.float32)
    fisher_matrix = np.hstack(fisher_views).astype(np.float32)
    acoustic      /= np.maximum(np.linalg.norm(acoustic, axis=1, keepdims=True), 1e-8)
    vlad_matrix   /= np.maximum(np.linalg.norm(vlad_matrix, axis=1, keepdims=True), 1e-8)
    fisher_matrix /= np.maximum(np.linalg.norm(fisher_matrix, axis=1, keepdims=True), 1e-8)

    assert np.isfinite(acoustic).all() and np.isfinite(vlad_matrix).all() and np.isfinite(fisher_matrix).all()
    np.savez_compressed(SHORTWIN_CACHE, acoustic=acoustic, vlad_matrix=vlad_matrix, fisher_matrix=fisher_matrix)
    del train_patches, test_patches, all_patches, boaw_views, vlad_views, fisher_views
    print(f"Mel short-window time: {time.time() - t0:.0f}s", flush=True)

vlad_kernel   = linear_kernel(vlad_matrix, vlad_matrix)
fisher_kernel = linear_kernel(fisher_matrix, fisher_matrix)
train_a, test_a = np.split(acoustic, [len(train_df)])
av, au = train_a[verified_mask], train_a[~verified_mask]
print("Mel BoAW:", acoustic.shape, "| VLAD:", vlad_matrix.shape, "| Fisher:", fisher_matrix.shape, flush=True)

## 3. Orthogonal CQT Fisher Vectors & fast PCA(512) compression for L2 Softmax

In [ ]:
print("Starting CQT Fisher + stats extraction...", flush=True)

CQT_KEY = hashlib.sha256(
    (",".join(train_df["id"].tolist()) + "|" + ",".join(test_df["id"].tolist()) + "|v1-cqt-60bins").encode()
).hexdigest()[:12]
CQT_CACHE = WORK / f"p4_cqt_v1_{CQT_KEY}.npz"

def extract_cqt_patches_and_stats(path):
    try:
        audio, rate = sf.read(str(path), dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if len(audio) < 4096:
            audio = np.pad(audio, (0, 4096 - len(audio)))
        hop = 512
        fmin = float(librosa.note_to_hz("C2"))
        max_bins = int(12 * np.floor(np.log2((rate * 0.45) / fmin)))
        n_bins = min(60, max(24, max_bins))
        C = np.abs(librosa.cqt(audio, sr=rate, hop_length=hop, fmin=fmin, n_bins=n_bins, bins_per_octave=12))
        logC = librosa.amplitude_to_db(C, ref=np.max).T.astype(np.float32)
        if n_bins < 60:
            logC = np.pad(logC, ((0, 0), (0, 60 - n_bins)), mode="edge")

        fe = logC.mean(axis=1)
        act = fe >= max(fe.max() - 45.0, float(np.percentile(fe, 25)))
        if act.sum() < 2:
            act = np.ones(len(logC), dtype=bool)
        logC_act = logC[act]

        stats = np.concatenate([
            logC_act.mean(axis=0),
            logC_act.std(axis=0),
            np.percentile(logC_act, 90, axis=0),
        ]).astype(np.float32)

        width = 16
        pad_len = max(0, width - len(logC))
        frames = np.pad(logC, ((0, pad_len), (0, 0)), mode="edge")
        fe_p = np.pad(fe, (0, pad_len), mode="edge")
        starts = list(range(0, len(frames) - width + 1, 8))
        if starts[-1] != len(frames) - width:
            starts.append(len(frames) - width)
        blocks = np.stack([frames[i:i + width] for i in starts])
        win_e = np.array([fe_p[i:i + width].mean() for i in starts], dtype=np.float32)
        m = win_e >= max(win_e.max() - 45.0, float(np.percentile(win_e, 25)))
        if m.sum() < 2:
            m = np.ones(len(blocks), dtype=bool)
        blocks = blocks[m]
        half = width // 2
        patches = np.hstack([
            blocks.mean(axis=1),
            blocks.std(axis=1),
            blocks[:, half:].mean(axis=1) - blocks[:, :half].mean(axis=1),
        ]).astype(np.float32)
        return patches, stats
    except Exception:
        return np.zeros((2, 180), dtype=np.float32), np.zeros(180, dtype=np.float32)

if CQT_CACHE.exists():
    z = np.load(CQT_CACHE)
    cqt_fisher = z["cqt_fisher"]
    cqt_stats  = z["cqt_stats"]
    print("Loaded cached CQT matrices:", cqt_fisher.shape, cqt_stats.shape, flush=True)
else:
    t0 = time.time()
    tr_res = Parallel(n_jobs=N_JOBS, batch_size=8, verbose=5)(
        delayed(extract_cqt_patches_and_stats)(TRAIN_AUDIO / f"{i}.wav") for i in train_df["id"]
    )
    te_res = Parallel(n_jobs=N_JOBS, batch_size=8, verbose=5)(
        delayed(extract_cqt_patches_and_stats)(TEST_AUDIO / f"{i}.wav") for i in test_df["id"]
    )
    all_cqt_patches = [r[0] for r in tr_res] + [r[0] for r in te_res]
    cqt_stats = np.vstack([r[1] for r in tr_res] + [r[1] for r in te_res]).astype(np.float32)
    cqt_stats = np.nan_to_num(cqt_stats, nan=0.0, posinf=0.0, neginf=0.0)

    rng = np.random.default_rng(123)
    samples = []
    for i in range(len(train_df)):
        fr = all_cqt_patches[i]
        cap = 36 if verified_mask[i] else 14
        samples.append(fr[rng.choice(len(fr), min(cap, len(fr)), replace=False)])
    pool = np.vstack(samples).astype(np.float32)
    sc_cqt = StandardScaler().fit(pool)
    pca_cqt = PCA(n_components=28, whiten=True, svd_solver="randomized", random_state=123)
    pool_p = pca_cqt.fit_transform(sc_cqt.transform(pool)).astype(np.float32)
    gmm_cqt = GaussianMixture(n_components=40, covariance_type="diag", reg_covar=0.01, max_iter=80, random_state=123)
    gmm_cqt.fit(pool_p)

    mu = gmm_cqt.means_.astype(np.float32)
    var = gmm_cqt.covariances_.astype(np.float32)
    sqrt_var = np.sqrt(var)
    sqrt_w = np.sqrt(gmm_cqt.weights_[:, None]).astype(np.float32)

    def encode_cqt_fv(fr):
        x = pca_cqt.transform(sc_cqt.transform(fr)).astype(np.float32)
        q = gmm_cqt.predict_proba(x).astype(np.float32)
        mass = q.sum(axis=0)[:, None]
        moments = q.T @ x
        norm = len(x) * sqrt_w
        first = (moments - mass * mu) / (norm * sqrt_var)
        second = ((q.T @ (x * x) - 2.0 * mu * moments + mass * (mu ** 2)) / var - mass) / (np.sqrt(2.0) * norm)
        v = np.concatenate([first.ravel(), second.ravel()])
        v = np.sign(v) * np.sqrt(np.abs(v))
        return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)

    cqt_fisher = np.stack([encode_cqt_fv(fr) for fr in all_cqt_patches]).astype(np.float32)
    assert np.isfinite(cqt_fisher).all() and np.isfinite(cqt_stats).all()
    np.savez_compressed(CQT_CACHE, cqt_fisher=cqt_fisher, cqt_stats=cqt_stats)
    del tr_res, te_res, all_cqt_patches
    print(f"CQT extraction + Fisher encoding time: {time.time() - t0:.0f}s", flush=True)

cqt_kernel = linear_kernel(cqt_fisher, cqt_fisher)
cqt_stats_tr, cqt_stats_te = np.split(cqt_stats, [len(train_df)])
cqt_stats_v, cqt_stats_u = cqt_stats_tr[verified_mask], cqt_stats_tr[~verified_mask]

# Unsupervised PCA(512) compression of joint [Mel-Fisher, Mel-VLAD, CQT-Fisher] -> ~25x faster LogisticRegression!
t_pca = time.time()
fv_joint_all = np.hstack([fisher_matrix, vlad_matrix, 0.85 * cqt_fisher]).astype(np.float32)
fv_joint_all /= np.maximum(np.linalg.norm(fv_joint_all, axis=1, keepdims=True), 1e-8)
pca_joint = PCA(n_components=512, svd_solver="randomized", random_state=42)
pca_joint.fit(fv_joint_all[:len(train_df)])  # fit strictly on train audio only
fv_512 = pca_joint.transform(fv_joint_all).astype(np.float32)
fv_512 /= np.maximum(np.linalg.norm(fv_512, axis=1, keepdims=True), 1e-8)
FV_tr, FV_te = np.split(fv_512, [len(train_df)])
FV_v, FV_u   = FV_tr[verified_mask], FV_tr[~verified_mask]
del fisher_matrix, vlad_matrix, cqt_fisher, fv_joint_all, fv_512
print(f"Joint FV+VLAD+CQT compressed to {FV_tr.shape[1]} dims in {time.time() - t_pca:.1f}s "
      f"(explained var: {pca_joint.explained_variance_ratio_.sum():.3f})", flush=True)

## 4. Active-trimmed `Modulation_v2`, Dense `PCA(256)` for fast XGBoost, and Fused Multi-Kernel

In [ ]:
print("Starting Modulation_v2 & Fused Multi-Kernel construction...", flush=True)

def modulation_features_v2(path):
    audio, rate = sf.read(str(path), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if audio.size == 0:
        audio = np.zeros(1024, dtype=np.float32)

    peak = float(np.max(np.abs(audio)))
    if peak > 1e-6:
        idx_nz = np.flatnonzero(np.abs(audio) >= peak * 0.0056)
        if len(idx_nz) >= max(1024, int(rate * 0.25)):
            audio = audio[idx_nz[0]:idx_nz[-1] + 1]

    width = 1024
    hop = max(1, round(rate * 0.010))
    audio = np.pad(audio, (width // 2, width // 2))
    frames = np.lib.stride_tricks.sliding_window_view(audio, width)[::hop]
    spectrum = np.abs(np.fft.rfft(frames * np.hanning(width), axis=1)) ** 2
    hz = np.fft.rfftfreq(width, 1.0 / rate)

    mel_edges = np.linspace(0, np.log1p(min(11000, rate / 2) / 700), 26)
    edges = 700 * np.expm1(mel_edges)
    bank = np.maximum(0, np.minimum(
        (hz[:, None] - edges[:-2]) / np.diff(edges)[:-1],
        (edges[2:] - hz[:, None]) / np.diff(edges)[1:],
    ))
    energy = spectrum @ bank
    envelope = np.log1p(100.0 * energy / max(float(energy.mean()), 1e-10))
    centered = envelope - envelope.mean(axis=0)
    scales = np.sqrt(np.mean(centered ** 2, axis=0))
    norm = centered / np.maximum(scales, 1e-6)
    corr = norm.T @ norm / max(len(norm), 1)

    freq, power = periodogram(
        centered, fs=rate / hop, axis=0,
        nfft=max(256, len(centered)), detrend=False, scaling="density"
    )
    bands = [(0.5, 2), (2, 4), (4, 8), (8, 16), (16, 32), (32, 50)]
    modulation = [
        np.log1p(power[(freq >= lo) & (freq < hi)].sum(axis=0) * freq[1])
        for lo, hi in bands
    ]
    sk = np.nan_to_num(skew(centered, axis=0), nan=0.0, posinf=0.0, neginf=0.0)
    ku = np.clip(np.nan_to_num(kurtosis(centered, axis=0), nan=0.0, posinf=0.0, neginf=0.0), -5.0, 30.0)
    mod_ratio = (modulation[0] + modulation[1] + modulation[2]) - (modulation[3] + modulation[4] + modulation[5])

    feat = np.concatenate([
        envelope.mean(axis=0), scales,
        corr[np.triu_indices(24, k=1)],
        *modulation, sk, ku, mod_ratio,
    ]).astype(np.float32)
    return np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)

def texture_matrix_v2(frame, folder):
    ids = frame["id"].to_numpy(dtype=str)
    key = hashlib.sha256(ids.tobytes()).hexdigest()[:12]
    path = WORK / f"modulation_v2_{folder.name}_{key}.npz"
    if path.exists():
        with np.load(path) as saved:
            if np.array_equal(saved["ids"], ids):
                return saved["features"]
    chunks = []
    for start in range(0, len(ids), 256):
        rows = Parallel(n_jobs=N_JOBS)(
            delayed(modulation_features_v2)(folder / f"{i}.wav") for i in ids[start:start + 256]
        )
        chunks.extend(rows)
        print(f"mod_v2 {folder.name}: {len(chunks)}/{len(ids)}", flush=True)
    features = np.stack(chunks).astype(np.float32)
    assert np.isfinite(features).all()
    np.savez_compressed(path, ids=ids, features=features)
    return features

texture_train = texture_matrix_v2(train_df, TRAIN_AUDIO)
texture_test  = texture_matrix_v2(test_df, TEST_AUDIO)
texture_v, texture_u = texture_train[verified_mask], texture_train[~verified_mask]

Xdense_tr = np.hstack([Xtr_rich, texture_train, train_a, cqt_stats_tr]).astype(np.float32)
Xdense_t  = np.hstack([XT_rich, texture_test, test_a, cqt_stats_te]).astype(np.float32)
Xdense_v, Xdense_u = Xdense_tr[verified_mask], Xdense_tr[~verified_mask]

# Unsupervised PCA(256) of standardized dense features for fast, decorrelated XGBoost (~6x faster!)
sc_dense_all = StandardScaler().fit(Xdense_tr)
pca_dense = PCA(n_components=256, svd_solver="randomized", random_state=42)
Xxgb_tr = pca_dense.fit_transform(sc_dense_all.transform(Xdense_tr)).astype(np.float32)
Xxgb_t  = pca_dense.transform(sc_dense_all.transform(Xdense_t)).astype(np.float32)
Xxgb_v, Xxgb_u = Xxgb_tr[verified_mask], Xxgb_tr[~verified_mask]

# Unsupervised positive-definite Fused Multi-Kernel across all 6 acoustic views
sc_rich_all = StandardScaler().fit(Xtr_rich)
X_rich_all_s = sc_rich_all.transform(np.vstack([Xtr_rich, Xte_rich])).astype(np.float32)
K_rich = rbf_kernel(X_rich_all_s, gamma=1.0 / X_rich_all_s.shape[1])

sc_mod_all = StandardScaler().fit(texture_train)
X_mod_all_s = sc_mod_all.transform(np.vstack([texture_train, texture_test])).astype(np.float32)
K_mod = rbf_kernel(X_mod_all_s, gamma=1.0 / X_mod_all_s.shape[1])

K_boaw = rbf_kernel(acoustic, gamma=1.0 / max(float(acoustic.var() * acoustic.shape[1]), 1e-6))

fused_kernel = (
    0.26 * fisher_kernel
    + 0.16 * vlad_kernel
    + 0.14 * cqt_kernel
    + 0.12 * K_boaw
    + 0.18 * K_rich
    + 0.14 * K_mod
).astype(np.float64)
d_inv = 1.0 / np.sqrt(np.maximum(np.diag(fused_kernel), 1e-12))
fused_kernel = fused_kernel * np.outer(d_inv, d_inv)
del X_rich_all_s, X_mod_all_s, K_rich, K_mod, K_boaw, Xdense_tr, Xxgb_tr
print("Modulation_v2:", texture_train.shape, "| Dense:", Xdense_v.shape,
      "| XGB PCA:", Xxgb_v.shape, "| Fused Kernel:", fused_kernel.shape, flush=True)

## 5. Stage 1: Dual-Teacher EM Noisy-Label Cleaning (Verified 5-View + Unchecked OOF)

In [ ]:
def _pred_aligned(model, X):
    out = np.zeros((len(X), K), dtype=np.float64)
    out[:, model.classes_] = model.predict_proba(X)
    return out

print("Computing 5-fold OOF unchecked-set regularized teacher (on 512-dim FV)...", flush=True)
t0 = time.time()
unc_splits = list(StratifiedKFold(5, shuffle=True, random_state=77).split(FV_u, yu))
def _fit_unc_fold(u_tr, u_va):
    w_u = balance_weights(yu[u_tr], np.ones(len(u_tr)), alpha=1.0)
    clf = LogisticRegression(C=6.0, max_iter=500)
    clf.fit(FV_u[u_tr], yu[u_tr], sample_weight=w_u)
    return _pred_aligned(clf, FV_u[u_va])

unc_res = Parallel(n_jobs=N_JOBS)(delayed(_fit_unc_fold)(u_tr, u_va) for u_tr, u_va in unc_splits)
Pu_unc_oof = np.zeros((len(yu), K), dtype=np.float64)
for (_, u_va), P_va in zip(unc_splits, unc_res):
    Pu_unc_oof[u_va] = P_va
Pu_unc_oof = sharpen(Pu_unc_oof, 1.35)
print(f"Unchecked OOF teacher done in {time.time() - t0:.1f}s (agreement top-1: {(Pu_unc_oof.argmax(1) == yu).mean():.3f})", flush=True)

def fit_predict_verified_teacher(tr_idx, va_idx=None):
    y_tr = yv[tr_idx]
    w_tr = balance_weights(y_tr, np.ones(len(tr_idx)), alpha=1.0)
    ids_tr = verified_ids[tr_idx]

    X_rich_tg = [XU_rich] + ([XV_rich[va_idx]] if va_idx is not None else [])
    X_mod_tg  = [texture_u] + ([texture_v[va_idx]] if va_idx is not None else [])
    X_fv_tg   = [FV_u] + ([FV_v[va_idx]] if va_idx is not None else [])
    X_den_tg  = [Xdense_u] + ([Xdense_v[va_idx]] if va_idx is not None else [])
    id_tg     = [unchecked_ids] + ([verified_ids[va_idx]] if va_idx is not None else [])

    m_rich = make_pipeline(StandardScaler(), SVC(C=5.0, kernel="rbf", gamma="scale", probability=True, random_state=42))
    m_rich.fit(XV_rich[tr_idx], y_tr, svc__sample_weight=w_tr)

    m_mod = make_pipeline(StandardScaler(), SVC(C=6.0, kernel="rbf", gamma="scale", probability=True, random_state=42))
    m_mod.fit(texture_v[tr_idx], y_tr, svc__sample_weight=w_tr)

    m_fv = LogisticRegression(C=14.0, max_iter=600)
    m_fv.fit(FV_v[tr_idx], y_tr, sample_weight=w_tr)

    m_mkl = SVC(kernel="precomputed", C=10.0, probability=True, random_state=42)
    m_mkl.fit(fused_kernel[np.ix_(ids_tr, ids_tr)], y_tr, sample_weight=w_tr)

    m_lda = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto", priors=np.full(K, 1.0 / K)))
    m_lda.fit(Xdense_v[tr_idx], y_tr)

    outputs = []
    for xr, xm, xfv, xd, tids in zip(X_rich_tg, X_mod_tg, X_fv_tg, X_den_tg, id_tg):
        p1 = _pred_aligned(m_rich, xr)
        p2 = _pred_aligned(m_mod, xm)
        p3 = _pred_aligned(m_fv, xfv)
        p4 = _pred_aligned(m_mkl, fused_kernel[np.ix_(tids, ids_tr)])
        p5 = _pred_aligned(m_lda, xd)
        p_lin = 0.20 * p1 + 0.15 * p2 + 0.25 * p3 + 0.28 * p4 + 0.12 * p5
        p_geo = normalize_probs(
            (np.clip(p1, 1e-6, None) ** 0.20)
            * (np.clip(p2, 1e-6, None) ** 0.15)
            * (np.clip(p3, 1e-6, None) ** 0.25)
            * (np.clip(p4, 1e-6, None) ** 0.28)
            * (np.clip(p5, 1e-6, None) ** 0.12)
        )
        outputs.append(0.5 * p_lin + 0.5 * p_geo)
    return outputs

def build_stage1(seed):
    splits = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(XV_rich, yv))
    res = Parallel(n_jobs=N_JOBS)(delayed(fit_predict_verified_teacher)(tr, va) for tr, va in splits)
    oof = np.zeros((len(yv), K), dtype=np.float64)
    Pu_list = []
    for (tr, va), (Pu, Pva) in zip(splits, res):
        oof[va] = Pva
        Pu_list.append(Pu)
    print(f"Stage-1 5-view verified teacher (seed {seed}) OOF plain MAP@3: {wmap3(yv, oof):.5f}", flush=True)
    return dict(splits=splits, Pu=Pu_list, oof=oof)

def clean_unchecked_dual_em(Pu_ver, temp, q_min=0.15, w_scale=0.60, floor_w=0.05, relabel_tau=0.88, relabel_w=0.25):
    P_ver_cal = sharpen(Pu_ver, temp)
    P_dual = normalize_probs(0.68 * P_ver_cal + 0.32 * Pu_unc_oof)
    _, Q, _ = em_noise(P_dual, yu, iters=50, prior_strength=5.0)
    idx = np.arange(len(yu))
    q_given = Q[idx, yu]
    best_cls = Q.argmax(axis=1)
    best_prob = Q[idx, best_cls]

    y_eff = yu.copy()
    w_eff = np.where(q_given >= q_min, w_scale * (q_given ** 0.75), floor_w)

    if relabel_tau is not None:
        relabel_mask = (q_given < 0.05) & (best_cls != yu) & (best_prob >= relabel_tau)
        y_eff[relabel_mask] = best_cls[relabel_mask]
        w_eff[relabel_mask] = relabel_w

    keep_hard = (q_given >= 0.45) | ((relabel_tau is not None) & (q_given < 0.05) & (best_cls != yu) & (best_prob >= relabel_tau))
    return y_eff, w_eff.astype(np.float64), keep_hard, q_given

ST1 = {seed: build_stage1(seed) for seed in SEEDS}

## 6. Stage 2: 4-Core Parallel 11-Model Diverse Ensemble (with live per-stage logging)

Runs all single-threaded SVMs + linear/LDA models concurrently across all 4 CPU cores, followed by multi-threaded RF/ET, LightGBM, and PCA(256)-XGBoost.

In [ ]:
MODEL_NAMES = [
    "frich_ens", "lgbm_dense", "xgb_dense", "lda_dense",
    "boaw_svm", "vlad_svm", "fisher_svm", "cqt_svm",
    "fused_mkl", "fv_logreg", "mod_svm",
]

def fit_predict_all_models_fast(tr_idx, y_u_eff, w_u_eff, keep_hard, va_idx=None, tag=""):
    y_all = np.concatenate([yv[tr_idx], y_u_eff])
    w_raw = np.concatenate([np.ones(len(tr_idx), dtype=np.float64), w_u_eff])
    w_bal = balance_weights(y_all, w_raw, alpha=1.0)

    keep_u = (w_u_eff >= 0.12)
    y_sub = np.concatenate([yv[tr_idx], y_u_eff[keep_u]])
    w_sub = balance_weights(y_sub, np.concatenate([np.ones(len(tr_idx)), w_u_eff[keep_u]]), alpha=1.0)
    y_lda = np.concatenate([yv[tr_idx], y_u_eff[keep_hard]])

    ids_tr = np.r_[verified_ids[tr_idx], unchecked_ids]
    ids_va = verified_ids[va_idx] if va_idx is not None else None

    Xr_tr = np.vstack([XV_rich[tr_idx], XU_rich[keep_u]])
    Xr_va = XV_rich[va_idx] if va_idx is not None else None
    Xb_tr = np.vstack([av[tr_idx], au])
    Xfv_tr = np.vstack([FV_v[tr_idx], FV_u])
    Xm_tr = np.vstack([texture_v[tr_idx], texture_u[keep_u]])
    Xd_lda = np.vstack([Xdense_v[tr_idx], Xdense_u[keep_hard]])

    # --- Batch A: Run all 11 single-threaded SVM / Linear / LDA fits in parallel across all 4 cores ---
    t_a = time.time()
    def _run_job(name):
        if name == "frich_svm3":
            m = make_pipeline(StandardScaler(), SVC(C=3.0, kernel="rbf", gamma="scale", probability=True, random_state=42))
            m.fit(Xr_tr, y_sub, svc__sample_weight=w_sub)
            return name, (_pred_aligned(m, Xr_va) if va_idx is not None else None, _pred_aligned(m, XT_rich))
        if name == "frich_svm10":
            m = make_pipeline(StandardScaler(), SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=42))
            m.fit(Xr_tr, y_sub, svc__sample_weight=w_sub)
            return name, (_pred_aligned(m, Xr_va) if va_idx is not None else None, _pred_aligned(m, XT_rich))
        if name == "frich_lr":
            m = make_pipeline(StandardScaler(), LogisticRegression(C=0.12, max_iter=1000))
            m.fit(Xr_tr, y_sub, logisticregression__sample_weight=w_sub)
            return name, (_pred_aligned(m, Xr_va) if va_idx is not None else None, _pred_aligned(m, XT_rich))
        if name == "boaw_svm":
            m = SVC(C=12.0, kernel="rbf", gamma="scale", probability=True, random_state=42)
            m.fit(Xb_tr, y_all, sample_weight=w_bal)
            return name, (_pred_aligned(m, av[va_idx]) if va_idx is not None else None, _pred_aligned(m, test_a))
        if name == "vlad_svm":
            m = SVC(kernel="precomputed", C=10.0, probability=True, random_state=42)
            m.fit(vlad_kernel[np.ix_(ids_tr, ids_tr)], y_all, sample_weight=w_bal)
            return name, (
                _pred_aligned(m, vlad_kernel[np.ix_(ids_va, ids_tr)]) if va_idx is not None else None,
                _pred_aligned(m, vlad_kernel[np.ix_(test_ids, ids_tr)]),
            )
        if name == "fisher_svm":
            m = SVC(kernel="precomputed", C=12.0, probability=True, random_state=42)
            m.fit(fisher_kernel[np.ix_(ids_tr, ids_tr)], y_all, sample_weight=w_bal)
            return name, (
                _pred_aligned(m, fisher_kernel[np.ix_(ids_va, ids_tr)]) if va_idx is not None else None,
                _pred_aligned(m, fisher_kernel[np.ix_(test_ids, ids_tr)]),
            )
        if name == "cqt_svm":
            m = SVC(kernel="precomputed", C=10.0, probability=True, random_state=42)
            m.fit(cqt_kernel[np.ix_(ids_tr, ids_tr)], y_all, sample_weight=w_bal)
            return name, (
                _pred_aligned(m, cqt_kernel[np.ix_(ids_va, ids_tr)]) if va_idx is not None else None,
                _pred_aligned(m, cqt_kernel[np.ix_(test_ids, ids_tr)]),
            )
        if name == "fused_mkl":
            m = SVC(kernel="precomputed", C=14.0, probability=True, random_state=42)
            m.fit(fused_kernel[np.ix_(ids_tr, ids_tr)], y_all, sample_weight=w_bal)
            return name, (
                _pred_aligned(m, fused_kernel[np.ix_(ids_va, ids_tr)]) if va_idx is not None else None,
                _pred_aligned(m, fused_kernel[np.ix_(test_ids, ids_tr)]),
            )
        if name == "fv_logreg":
            m = LogisticRegression(C=20.0, max_iter=600)
            m.fit(Xfv_tr, y_all, sample_weight=w_bal)
            return name, (_pred_aligned(m, FV_v[va_idx]) if va_idx is not None else None, _pred_aligned(m, FV_te))
        if name == "mod_svm":
            m = make_pipeline(StandardScaler(), SVC(C=8.0, kernel="rbf", gamma="scale", probability=True, random_state=42))
            m.fit(Xm_tr, y_sub, svc__sample_weight=w_sub)
            return name, (_pred_aligned(m, texture_v[va_idx]) if va_idx is not None else None, _pred_aligned(m, texture_test))
        if name == "lda_dense":
            m = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto", priors=np.full(K, 1.0 / K)))
            m.fit(Xd_lda, y_lda)
            return name, (_pred_aligned(m, Xdense_v[va_idx]) if va_idx is not None else None, _pred_aligned(m, Xdense_t))
        raise ValueError(name)

    parallel_jobs = [
        "fused_mkl", "fisher_svm", "vlad_svm", "cqt_svm",
        "boaw_svm", "mod_svm", "frich_svm3", "frich_svm10",
        "frich_lr", "fv_logreg", "lda_dense",
    ]
    # Use threading-safe loky backend across all CPU cores
    res_a = dict(Parallel(n_jobs=N_JOBS, prefer="threads")(delayed(_run_job)(nm) for nm in parallel_jobs))
    dt_a = time.time() - t_a

    # --- Batch B: Multi-threaded tree ensembles (RF + ET, LightGBM, PCA-XGBoost) ---
    t_rf = time.time()
    rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=N_JOBS)
    et = ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=N_JOBS)
    rf.fit(Xr_tr, y_sub, sample_weight=w_sub)
    et.fit(Xr_tr, y_sub, sample_weight=w_sub)
    p_va_f = None
    if va_idx is not None:
        p_va_f = (
            0.25 * res_a["frich_svm3"][0] + 0.20 * res_a["frich_svm10"][0] + 0.15 * res_a["frich_lr"][0]
            + 0.20 * _pred_aligned(rf, Xr_va) + 0.20 * _pred_aligned(et, Xr_va)
        )
    p_te_f = (
        0.25 * res_a["frich_svm3"][1] + 0.20 * res_a["frich_svm10"][1] + 0.15 * res_a["frich_lr"][1]
        + 0.20 * _pred_aligned(rf, XT_rich) + 0.20 * _pred_aligned(et, XT_rich)
    )
    dt_rf = time.time() - t_rf

    t_lgb = time.time()
    Xd_tr = np.vstack([Xdense_v[tr_idx], Xdense_u[keep_u]])
    lgb = LGBMClassifier(
        objective="multiclass", num_class=K,
        n_estimators=350, learning_rate=0.05, num_leaves=15, max_bin=63,
        min_child_samples=18, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.18, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, n_jobs=N_JOBS, verbose=-1,
    )
    lgb.fit(Xd_tr, y_sub, sample_weight=w_sub)
    p_lgb = (_pred_aligned(lgb, Xdense_v[va_idx]) if va_idx is not None else None, _pred_aligned(lgb, Xdense_t))
    dt_lgb = time.time() - t_lgb

    t_xgb = time.time()
    Xx_tr = np.vstack([Xxgb_v[tr_idx], Xxgb_u[keep_u]])
    xgb = XGBClassifier(
        objective="multi:softprob", num_class=K,
        tree_method="hist", max_bin=64, n_estimators=260, learning_rate=0.06,
        max_depth=4, min_child_weight=4, subsample=0.8,
        colsample_bytree=0.35, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, n_jobs=N_JOBS, eval_metric="mlogloss", verbosity=0,
    )
    xgb.fit(Xx_tr, y_sub, sample_weight=w_sub)
    p_xgb = (_pred_aligned(xgb, Xxgb_v[va_idx]) if va_idx is not None else None, _pred_aligned(xgb, Xxgb_t))
    dt_xgb = time.time() - t_xgb

    print(f"  [{tag}] SVMs+Lin+LDA(4-core): {dt_a:.0f}s | RF+ET: {dt_rf:.0f}s | LGBM: {dt_lgb:.0f}s | XGB(PCA): {dt_xgb:.0f}s", flush=True)

    return {
        "frich_ens":  (p_va_f, p_te_f),
        "lgbm_dense": p_lgb,
        "xgb_dense":  p_xgb,
        "lda_dense":  res_a["lda_dense"],
        "boaw_svm":   res_a["boaw_svm"],
        "vlad_svm":   res_a["vlad_svm"],
        "fisher_svm": res_a["fisher_svm"],
        "cqt_svm":    res_a["cqt_svm"],
        "fused_mkl":  res_a["fused_mkl"],
        "fv_logreg":  res_a["fv_logreg"],
        "mod_svm":    res_a["mod_svm"],
    }

t0 = time.time()
OOF_MODELS = {seed: {m: np.zeros((len(yv), K), dtype=np.float64) for m in MODEL_NAMES} for seed in SEEDS}
TEST_BAG   = {m: np.zeros((len(test_df), K), dtype=np.float64) for m in MODEL_NAMES}

for seed in SEEDS:
    st = ST1[seed]
    print(f"\n=== Stage 2 CV: Seed {seed} ===", flush=True)
    for fold, ((tr, va), Pu) in enumerate(zip(st["splits"], st["Pu"]), 1):
        t_fold = time.time()
        temp_tr = fit_temp(st["oof"][tr], yv[tr], w_unif[tr])
        y_u_eff, w_u_eff, keep_hard, _ = clean_unchecked_dual_em(Pu, temp_tr)
        preds = fit_predict_all_models_fast(tr, y_u_eff, w_u_eff, keep_hard, va_idx=va, tag=f"S{seed} F{fold}/5")
        for m in MODEL_NAMES:
            OOF_MODELS[seed][m][va] = preds[m][0]
            TEST_BAG[m] += preds[m][1] / (len(SEEDS) * 5)
        print(
            f"  Seed {seed} fold {fold}/5 total: {time.time() - t_fold:.0f}s "
            f"(kept={(w_u_eff >= 0.12).sum()}, relabeled={(y_u_eff != yu).sum()})",
            flush=True,
        )

print(f"\nStage 2 CV total time: {time.time() - t0:.0f}s", flush=True)
print("\nIndividual model OOF plain MAP@3:", flush=True)
for m in MODEL_NAMES:
    scores = [wmap3(yv, OOF_MODELS[s][m]) for s in SEEDS]
    print(f"  {m:12s}: seed42={scores[0]:.5f} | seed913={scores[1]:.5f} | mean={np.mean(scores):.5f}", flush=True)

## 7. Cross-fitted per-model calibration, regularized stacking, and acoustic graph label propagation

In [ ]:
def blend_calibrated(P_list, weights, lam):
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    p_lin = sum(wi * Pi for wi, Pi in zip(w, P_list))
    log_p = sum(wi * np.log(np.clip(Pi, 1e-9, None)) for wi, Pi in zip(w, P_list))
    log_p -= log_p.max(axis=1, keepdims=True)
    p_geo = np.exp(log_p)
    p_geo /= p_geo.sum(axis=1, keepdims=True)
    return (1.0 - lam) * p_lin + lam * p_geo

def fit_blend_params(oof_dict, tr_idx, reg_l2=0.04):
    y_tr = yv[tr_idx]
    w_tr = w_unif[tr_idx]
    idx = np.arange(len(y_tr))
    M = len(MODEL_NAMES)

    temps = [fit_temp(oof_dict[m][tr_idx], y_tr, w_tr) for m in MODEL_NAMES]
    P_cal = [sharpen(oof_dict[m][tr_idx], t) for m, t in zip(MODEL_NAMES, temps)]

    w0 = np.full(M, 1.0 / M)
    def obj(params):
        w = params[:M]
        lam = params[M]
        P = blend_calibrated(P_cal, w, lam)
        nll = -float(np.average(np.log(np.clip(P[idx, y_tr], 1e-12, None)), weights=w_tr))
        return nll + reg_l2 * float(np.sum((w - w0) ** 2))

    bounds = [(0.0, 0.45)] * M + [(0.0, 0.85)]
    cons = {"type": "eq", "fun": lambda p: np.sum(p[:M]) - 1.0}
    init = np.concatenate([w0, [0.40]])
    res = minimize(obj, init, method="SLSQP", bounds=bounds, constraints=cons, options={"maxiter": 200, "ftol": 1e-7})
    w_opt = np.clip(res.x[:M], 0.0, None)
    w_opt /= w_opt.sum()
    lam_opt = float(np.clip(res.x[M], 0.0, 1.0))

    P_tr_blend = blend_calibrated(P_cal, w_opt, lam_opt)
    t_ens = fit_temp(P_tr_blend, y_tr, w_tr)
    return temps, w_opt, lam_opt, t_ens

OOF_ENS = {}
LEARNED_PARAMS = []
DIFF_CACHE = {seed: [] for seed in SEEDS}

for seed in SEEDS:
    splits = ST1[seed]["splits"]
    oof_ens = np.zeros((len(yv), K), dtype=np.float64)
    for fold_idx, (tr, va) in enumerate(splits):
        temps, w_opt, lam_opt, t_ens = fit_blend_params(OOF_MODELS[seed], tr, reg_l2=0.04)
        P_va_cal = [sharpen(OOF_MODELS[seed][m][va], t) for m, t in zip(MODEL_NAMES, temps)]
        oof_ens[va] = sharpen(blend_calibrated(P_va_cal, w_opt, lam_opt), t_ens)
        LEARNED_PARAMS.append((temps, w_opt, lam_opt, t_ens))

        temp_tr = fit_temp(ST1[seed]["oof"][tr], yv[tr], w_unif[tr])
        P_dual_u = normalize_probs(0.68 * sharpen(ST1[seed]["Pu"][fold_idx], temp_tr) + 0.32 * Pu_unc_oof)
        _, Q_u, _ = em_noise(P_dual_u, yu, iters=50, prior_strength=5.0)
        P_src = np.vstack([np.eye(K)[yv[tr]], Q_u])
        src_ids = np.r_[verified_ids[tr], unchecked_ids]
        top_idx, w_knn = build_knn_diffusion_matrix(fused_kernel, verified_ids[va], src_ids, k_nn=15, tau=0.12)
        DIFF_CACHE[seed].append((va, top_idx, w_knn, P_src))

    OOF_ENS[seed] = oof_ens
    print(f"Seed {seed} stacked OOF plain MAP@3 (before graph diffusion): {wmap3(yv, oof_ens):.5f}", flush=True)

alpha_grid = [0.0, 0.05, 0.10, 0.15, 0.20]
alpha_scores = {}
for alpha in alpha_grid:
    s_list = []
    for seed in SEEDS:
        P_d = OOF_ENS[seed].copy()
        for va, top_idx, w_knn, P_src in DIFF_CACHE[seed]:
            P_d[va] = apply_graph_diffusion(P_d[va], P_src, top_idx, w_knn, alpha=alpha)
        s_list.append(wmap3(yv, P_d))
    alpha_scores[alpha] = float(np.mean(s_list))

BEST_ALPHA = max(alpha_scores, key=alpha_scores.get)
print("Graph diffusion CV plain MAP@3 by alpha:", {k: round(v, 5) for k, v in alpha_scores.items()}, "-> selected:", BEST_ALPHA, flush=True)

OOF_FINAL = {}
for seed in SEEDS:
    P_d = OOF_ENS[seed].copy()
    for va, top_idx, w_knn, P_src in DIFF_CACHE[seed]:
        P_d[va] = apply_graph_diffusion(P_d[va], P_src, top_idx, w_knn, alpha=BEST_ALPHA)
    OOF_FINAL[seed] = P_d
    rep = cv_report(P_d, ST1[seed]["splits"])
    print(f"\n=== Final CV Report (Seed {seed}) ===", flush=True)
    print(f"  plain={rep['plain']:.5f} | w_raw={rep['w_raw']:.5f} | u_raw={rep['u_raw']:.5f}", flush=True)
    print(f"  w_cal={rep['w_cal']:.5f} | w_em0.25={rep['w_em0.25']:.5f} | w_em0.50={rep['w_em0.50']:.5f} | w_em0.75={rep['w_em0.75']:.5f}", flush=True)
    print(f"  u_cal={rep['u_cal']:.5f} | u_em0.25={rep['u_em0.25']:.5f} | u_em0.50={rep['u_em0.50']:.5f} | u_em0.75={rep['u_em0.75']:.5f}", flush=True)

MEAN_TEMPS = np.mean([p[0] for p in LEARNED_PARAMS], axis=0)
MEAN_W     = np.mean([p[1] for p in LEARNED_PARAMS], axis=0)
MEAN_W    /= MEAN_W.sum()
MEAN_LAM   = float(np.mean([p[2] for p in LEARNED_PARAMS]))
MEAN_T_ENS = float(np.mean([p[3] for p in LEARNED_PARAMS]))

print("\nLearned ensemble weights (10-fold mean):", flush=True)
for m, t, w in zip(MODEL_NAMES, MEAN_TEMPS, MEAN_W):
    print(f"  {m:12s}: weight={w:.4f} | temp={t:.3f}", flush=True)
print(f"  log-pool mix lambda={MEAN_LAM:.3f} | final ens temp={MEAN_T_ENS:.3f}", flush=True)

em_scores = {
    pw: float(np.mean([combine_score(cv_report(OOF_FINAL[s], ST1[s]["splits"]), pw) for s in SEEDS]))
    for pw in (0.0, 0.25, 0.50, 0.75, 1.0)
}
BEST_POWER = max(em_scores, key=em_scores.get)
print("CV combined score by EM power:", {k: round(v, 5) for k, v in em_scores.items()}, "-> selected:", BEST_POWER, flush=True)

## 8. Full-data retraining, CV + full bagging, graph diffusion, and submission

In [ ]:
print("Starting full-data final training...", flush=True)
t0 = time.time()

all_verified = np.arange(len(yv))
teacher_full_temp = float(np.mean([fit_temp(ST1[s]["oof"], yv, w_unif) for s in SEEDS]))
Pu_full = fit_predict_verified_teacher(all_verified, va_idx=None)[0]
y_u_full, w_u_full, keep_hard_full, _ = clean_unchecked_dual_em(Pu_full, teacher_full_temp)

print(f"Full dual-teacher EM cleaning: kept={(w_u_full >= 0.12).sum()}, relabeled={(y_u_full != yu).sum()}", flush=True)
full_preds = fit_predict_all_models_fast(all_verified, y_u_full, w_u_full, keep_hard_full, va_idx=None, tag="FULL")
print(f"Full-data Stage 2 training time: {time.time() - t0:.0f}s", flush=True)

P_test_cal_list = []
for m, t in zip(MODEL_NAMES, MEAN_TEMPS):
    p_m = 0.50 * TEST_BAG[m] + 0.50 * full_preds[m][1]
    P_test_cal_list.append(sharpen(p_m, t))

P_test_blend = sharpen(blend_calibrated(P_test_cal_list, MEAN_W, MEAN_LAM), MEAN_T_ENS)

if BEST_ALPHA > 0.0:
    P_dual_full = normalize_probs(0.68 * sharpen(Pu_full, teacher_full_temp) + 0.32 * Pu_unc_oof)
    _, Q_u_full, _ = em_noise(P_dual_full, yu, iters=50, prior_strength=5.0)
    P_src_full = np.vstack([np.eye(K)[yv], Q_u_full])
    src_ids_full = np.r_[verified_ids, unchecked_ids]
    top_idx_te, w_knn_te = build_knn_diffusion_matrix(fused_kernel, test_ids, src_ids_full, k_nn=15, tau=0.12)
    P_test_blend = apply_graph_diffusion(P_test_blend, P_src_full, top_idx_te, w_knn_te, alpha=BEST_ALPHA)

P_test_final, pi_test_hat = finalize_probs(P_test_blend, temp=1.0, power=BEST_POWER, pi_train=None)

sub = write_submission(P_test_final, "p4_fast_final_submission.csv")

source = WORK / "model_files" / "p4_fast_final_submission.csv"
destination = WORK / "submission.csv"
assert source.exists(), "Prediction file not found!"
shutil.copy(source, destination)
print("Submission file created:", destination, "| exists:", destination.exists(), flush=True)

top1_counts = pd.Series(sub["label"].str.split().str[0]).value_counts()
print(f"Unique top-1 classes predicted: {len(top1_counts)}/{K} | min={top1_counts.min()} | max={top1_counts.max()}", flush=True)
print(sub.head(10))